# The most accurate useless detector you will ever build

MichAl Academy, lesson 2.7.

Run each cell with **Shift+Enter**.

Every accuracy figure in this track so far came from a balanced problem. Almost
nothing you will be asked to detect is balanced, and this notebook shows what
happens to each of the standard metrics when it is not.

The result to look for: a model with an ROC-AUC of 0.986 that catches nothing
at all, and an accuracy identical to a model that does nothing.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score,
                             average_precision_score)


def fit_and_score(X, y, seed=0):
    """Fit on 70%, return the held-back labels and the model's scores for them."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y
    )
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    model.fit(X_train, y_train)
    return y_test, model.predict_proba(X_test)[:, 1]


def report(y_true, scores, threshold=0.5):
    alert = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, alert, labels=[0, 1]).ravel()
    return {
        "base rate": y_true.mean(),
        "accuracy": accuracy_score(y_true, alert),
        "never alert": 1 - y_true.mean(),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision_score(y_true, alert, zero_division=0),
        "recall": recall_score(y_true, alert, zero_division=0),
        "f1": f1_score(y_true, alert, zero_division=0),
        "roc auc": roc_auc_score(y_true, scores),
        "avg precision": average_precision_score(y_true, scores),
    }


## 1. One in ten

The digits from lesson 2.2, asked a yes-or-no question: is this an 8? There are
ten digits, so about a tenth of the images are 8s and this is a mildly
imbalanced problem rather than a severe one.


In [ ]:
digits = load_digits()
is_eight = (digits.target == 8).astype(int)
print(f"{is_eight.sum()} of {len(is_eight)} images are 8s, a base rate of {is_eight.mean():.4f}")

y_test, scores = fit_and_score(digits.data, is_eight)
natural = report(y_test, scores)
for k, v in natural.items():
    print(f"  {k:>14}  {v:.4f}" if isinstance(v, float) else f"  {k:>14}  {v}")


Accuracy 0.9630, which sounds excellent, so put it beside the only number that
gives it meaning: a model that never alerts at all scores **0.9037**.

The entire achievement of this model, measured in accuracy, is six points over
answering "no" to everything. Accuracy is a share of a total, and when one class
is 90% of the total, the total is telling you about that class and almost
nothing about the one you care about.

The four numbers underneath are the ones that mean something.


## 2. Where you put the cut decides which mistake you make

The model returns a probability, so 0.5 is a choice rather than a fact. Sweep it.


In [ ]:
print(f"{'threshold':>10} {'caught':>7} {'false':>6} {'missed':>7} {'precision':>10} {'recall':>7} {'f1':>6}")
for t in (0.02, 0.05, 0.10, 0.20, 0.50, 0.80, 0.95):
    r = report(y_test, scores, t)
    print(f"{t:>10.2f} {r['tp']:>7} {r['fp']:>6} {r['fn']:>7} "
          f"{r['precision']:>10.3f} {r['recall']:>7.3f} {r['f1']:>6.3f}")


Read the two middle columns against each other. There is no setting that is
simply better. At 0.05 the detector catches 94% of the 8s and half its alerts
are wrong. At 0.80 it is never wrong and misses 44% of them.

Which of those is correct depends entirely on what each mistake costs you, and
that is not a question machine learning can answer. Lesson 2.8 is about
answering it deliberately instead of leaving the threshold at whatever the
library defaulted to.


## 3. Now make it rare

Keep every image that is not an 8, and throw away all but eighteen of the 8s.
The detector is untouched. Only the world changed.


In [ ]:
rng = np.random.default_rng(0)
not_eight = np.where(is_eight == 0)[0]
some_eights = rng.choice(np.where(is_eight == 1)[0], 18, replace=False)
rare = np.concatenate([not_eight, some_eights])

y_rare, scores_rare = fit_and_score(digits.data[rare], is_eight[rare])
rarified = report(y_rare, scores_rare)

comparison = pd.DataFrame({"one in ten": natural, "one in a hundred": rarified})
print(comparison.round(4).to_string())


Sit with that column.

**Accuracy went up.** 0.9899, better than the 0.9630 we were pleased with.

**And it is exactly the accuracy of never alerting.** Both 0.9899, to four
decimal places, because the model raised no alerts at all: zero caught, zero
false alarms, five missed.

**Precision, recall and F1 are all zero.** They are the only metrics in the
table telling the truth.

**And the ROC-AUC is 0.986**, up from 0.980. By that number the model looks
slightly better than it did.


## 4. Why ROC-AUC lied and average precision did not

ROC-AUC is built from the true positive rate and the **false positive rate**,
and a false positive rate is a share of the negatives. When negatives are 99% of
your data, adding thousands of them barely moves that denominator, so the number
holds up while the analyst's experience collapses.

Average precision is built from precision, which is a share of the **alerts**,
and that is exactly the quantity that falls apart.


In [ ]:
print(f"{'metric':>18} {'one in ten':>11} {'one in a hundred':>17}")
for m in ("roc auc", "avg precision", "precision", "recall"):
    print(f"{m:>18} {natural[m]:>11.3f} {rarified[m]:>17.3f}")


ROC-AUC moved by +0.006. Average precision halved.

Saito and Rehmsmeier made this case in 2015: on imbalanced data the
precision-recall view "can provide the viewer with an accurate prediction of
future classification performance" where the ROC view does not. scikit-learn's
own guidance notes that a random model's average precision equals the fraction
of positives, so on a 1% problem random scores 0.01, which gives you the floor
to judge 0.446 against.

One caution about this particular experiment: the rarified test set contains
only five real 8s, so its individual numbers are noisy. The pattern is not. It
is arithmetic about denominators rather than a property of this dataset.


## 5. Translate it into a working day

The last step is the one that changes decisions. Take the rates the model
actually achieved and apply them to a volume.


In [ ]:
EVENTS_PER_DAY = 50_000

tpr = natural["recall"]
fpr = natural["fp"] / (natural["fp"] + natural["tn"])
print(f"measured at threshold 0.5:  catches {tpr:.1%} of real cases, "
      f"misfires on {fpr:.2%} of ordinary ones")
print()
print(f"{'base rate':>12} {'real/day':>9} {'alerts/day':>11} {'wasted':>8} {'worth reading':>14}")
for label, rate in [("1 in 10", 0.1), ("1 in 100", 0.01), ("1 in 1000", 0.001), ("1 in 10000", 0.0001)]:
    real = EVENTS_PER_DAY * rate
    caught = tpr * real
    wasted = fpr * (EVENTS_PER_DAY - real)
    total = caught + wasted
    share = caught / total if total else 0
    print(f"{label:>12} {real:>9.0f} {total:>11.0f} {wasted:>8.0f} {share:>13.1%}")


Same detector on every row, and look at the last column fall: 90%, 46%, 8%,
0.8%.

At one in ten this is a useful tool. At one in a hundred, more than half of every
alert is a waste of somebody's time. At one in ten thousand, 99% of what reaches
a person is noise, and no retraining fixes it, because nothing about the model
is wrong. It still catches 69% of real cases and still misfires on only 0.82% of
ordinary ones. Those are the same two numbers throughout.

This is the calculation to run **before** building anything. If the base rate
and the volume mean the output is unusable at your model's plausible error
rates, that is worth knowing in week one rather than at deployment.


## What to take from this

| Metric | What it is a share of | Survives imbalance? |
|---|---|---|
| Accuracy | Everything | No. Compare it against never alerting |
| Recall | The real cases | Yes. A property of the detector |
| Precision | The alerts raised | Yes, and it is what collapses |
| F1 | Precision and recall combined | Yes |
| ROC-AUC | Negatives, via the false positive rate | **No. It barely moved while the model became useless** |
| Average precision | The alerts raised | Yes |

Three habits.

**Never read an accuracy without the base rate beside it.** "98.99% accurate"
and "identical to doing nothing" were the same model.

**Report precision and recall, or a curve, never a single number.** A single
number hides which of the two mistakes you chose to make.

**Do the alerts-per-day arithmetic before you build.** It takes two minutes and
it is the only calculation here that has ever cancelled a project, which is
sometimes the correct outcome.
